# DS Main Project 
#### Student Name: Muthupandian S
#### Batch: DSAN01
#### Project Title: Next Word Prediction Using LSTM
#### Project Category: Deep Learning / NLP / LSTM 
#### Submission Date:

<font color="197AA6" size=5px><b>Stage 1-Data selection , EDA, data cleaning ,data preprocessing</b></font>

In [75]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import requests
import pandas as pd
import time
import re
import ast

# Dataset Selection & Understanding

Dataset Source: "https://datasets-server.huggingface.co/rows"

In [76]:
url = "https://datasets-server.huggingface.co/rows"

recipes = []

for offset in range(0, 5000, 100):

    params = {
        "dataset": "Mahimas/recipenlg",
        "config": "default",
        "split": "train",
        "offset": offset,
        "length": 100
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("Error:", response.status_code)
        break

    data = response.json()

    for item in data["rows"]:
        recipes.append(item["row"])

    print(f"Downloaded: {len(recipes)}")

    time.sleep(0.2)

Downloaded: 100
Downloaded: 200
Downloaded: 300
Downloaded: 400
Downloaded: 500
Downloaded: 600
Downloaded: 700
Downloaded: 800
Downloaded: 900
Downloaded: 1000
Downloaded: 1100
Downloaded: 1200
Downloaded: 1300
Downloaded: 1400
Downloaded: 1500
Downloaded: 1600
Downloaded: 1700
Downloaded: 1800
Downloaded: 1900
Downloaded: 2000
Downloaded: 2100
Downloaded: 2200
Downloaded: 2300
Downloaded: 2400
Downloaded: 2500
Downloaded: 2600
Downloaded: 2700
Downloaded: 2800
Downloaded: 2900
Downloaded: 3000
Downloaded: 3100
Downloaded: 3200
Downloaded: 3300
Downloaded: 3400
Error: 502


In [77]:
recipes

[{'Unnamed: 0': 0,
  'title': 'No-Bake Nut Cookies',
  'ingredients': '["1 c. firmly packed brown sugar", "1/2 c. evaporated milk", "1/2 tsp. vanilla", "1/2 c. broken nuts (pecans)", "2 Tbsp. butter or margarine", "3 1/2 c. bite size shredded rice biscuits"]',
  'directions': '["In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.", "Stir over medium heat until mixture bubbles all over top.", "Boil and stir 5 minutes more. Take off heat.", "Stir in vanilla and cereal; mix well.", "Using 2 teaspoons, drop and shape into 30 clusters on wax paper.", "Let stand until firm, about 30 minutes."]',
  'link': 'www.cookbooks.com/Recipe-Details.aspx?id=44874',
  'source': 'Gathered',
  'NER': '["brown sugar", "milk", "vanilla", "nuts", "butter", "bite size shredded rice biscuits"]'},
 {'Unnamed: 0': 1,
  'title': "Jewell Ball'S Chicken",
  'ingredients': '["1 small jar chipped beef, cut up", "4 boned chicken breasts", "1 can cream of mushroom soup", "1 cart

In [78]:
df = pd.DataFrame(recipes)

In [79]:
df.to_csv('dataset.csv')

In [80]:
df.head()

,Unnamed: 0,title,ingredients,directions,link,source,NER
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom..."
2,2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar..."
3,3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo..."
4,4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu..."


In [81]:
df.tail()

,Unnamed: 0,title,ingredients,directions,link,source,NER
3395,3395,Noodle-Rice Casserole,"[""1/4 c. butter"", ""2 c. water"", ""3/4 oz. angel...","[""Melt butter in large skillet."", ""Add spaghet...",www.cookbooks.com/Recipe-Details.aspx?id=1048306,Gathered,"[""butter"", ""water"", ""hair spaghetti"", ""long gr..."
3396,3396,Apple Dip,"[""6 Granny Smith apples, sliced"", ""12 oz. crea...","[""Arrange apple slices around plate. Combine c...",www.cookbooks.com/Recipe-Details.aspx?id=558365,Gathered,"[""apples"", ""cream cheese"", ""sugar"", ""brown sug..."
3397,3397,3 Hour Sunday Rolls,"[""2 pkg. yeast"", ""2 whole eggs"", ""1/4 c. Wesso...","[""Mix the above 6 ingredients together."", ""Add...",www.cookbooks.com/Recipe-Details.aspx?id=963022,Gathered,"[""yeast"", ""eggs"", ""Wesson oil"", ""water"", ""suga..."
3398,3398,Slomgolian,"[""1 lb. ground beef"", ""1/2 c. chopped onion"", ...","[""Brown ground beef."", ""Saute onion and bell p...",www.cookbooks.com/Recipe-Details.aspx?id=8391,Gathered,"[""ground beef"", ""onion"", ""bell pepper"", ""mushr..."
3399,3399,Orange Sherbet Salad,"[""2 (3 oz.) pkg. orange gelatin"", ""2 c. hot wa...","[""Using juice from the mandarin oranges as par...",www.cookbooks.com/Recipe-Details.aspx?id=316249,Gathered,"[""orange gelatin"", ""water"", ""mandarin oranges""..."


In [82]:
df.shape

(3400, 7)

In [83]:
df.size

23800

In [84]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3400 entries, 0 to 3399
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Unnamed: 0   3400 non-null   int64
 1   title        3400 non-null   str  
 2   ingredients  3400 non-null   str  
 3   directions   3400 non-null   str  
 4   link         3400 non-null   str  
 5   source       3400 non-null   str  
 6   NER          3400 non-null   str  
dtypes: int64(1), str(6)
memory usage: 2.2 MB


In [85]:
food_recipes = df["directions"].astype(str)

In [86]:
def convert_to_text(x):
    try:
        return " ".join(ast.literal_eval(x))
    except:
        return str(x)

recipes_txt = df["directions"].apply(convert_to_text)

In [87]:
recipes_txt

0       In a heavy 2-quart saucepan, mix brown sugar, ...
1       Place chipped beef on bottom of baking dish. P...
2       In a slow cooker, combine all ingredients. Cov...
3       Boil and debone chicken. Put bite size pieces ...
4       Combine first four ingredients and press in 13...
                              ...                        
3395    Melt butter in large skillet. Add spaghetti; c...
3396    Arrange apple slices around plate. Combine cre...
3397    Mix the above 6 ingredients together. Add enou...
3398    Brown ground beef. Saute onion and bell pepper...
3399    Using juice from the mandarin oranges as part ...
Name: directions, Length: 3400, dtype: str

In [88]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

recipes_cleaned = recipes_txt.apply(clean_text)

In [89]:
recipes_cleaned

0       in a heavy 2quart saucepan mix brown sugar nut...
1       place chipped beef on bottom of baking dish pl...
2       in a slow cooker combine all ingredients cover...
3       boil and debone chicken put bite size pieces i...
4       combine first four ingredients and press in 13...
                              ...                        
3395    melt butter in large skillet add spaghetti coo...
3396    arrange apple slices around plate combine crea...
3397    mix the above 6 ingredients together add enoug...
3398    brown ground beef saute onion and bell pepper ...
3399    using juice from the mandarin oranges as part ...
Name: directions, Length: 3400, dtype: str

In [90]:
corpus_text = " ".join(recipes_cleaned)
print(corpus_text)

in a heavy 2quart saucepan mix brown sugar nuts evaporated milk and butter or margarine stir over medium heat until mixture bubbles all over top boil and stir 5 minutes more take off heat stir in vanilla and cereal mix well using 2 teaspoons drop and shape into 30 clusters on wax paper let stand until firm about 30 minutes place chipped beef on bottom of baking dish place chicken on top of beef mix soup and cream together pour over chicken bake uncovered at 275 for 3 hours in a slow cooker combine all ingredients cover and cook on low for 4 hours or until heated through and cheese is melted stir well before serving yields 6 servings boil and debone chicken put bite size pieces in average size square casserole dish pour gravy and cream of mushroom soup over chicken level make stuffing according to instructions on box do not make too moist put stuffing on top of chicken and gravy level sprinkle shredded cheese on top and bake at 350 for approximately 20 minutes or until golden and bubbly

In [91]:
corpus_text[:2000]

'in a heavy 2quart saucepan mix brown sugar nuts evaporated milk and butter or margarine stir over medium heat until mixture bubbles all over top boil and stir 5 minutes more take off heat stir in vanilla and cereal mix well using 2 teaspoons drop and shape into 30 clusters on wax paper let stand until firm about 30 minutes place chipped beef on bottom of baking dish place chicken on top of beef mix soup and cream together pour over chicken bake uncovered at 275 for 3 hours in a slow cooker combine all ingredients cover and cook on low for 4 hours or until heated through and cheese is melted stir well before serving yields 6 servings boil and debone chicken put bite size pieces in average size square casserole dish pour gravy and cream of mushroom soup over chicken level make stuffing according to instructions on box do not make too moist put stuffing on top of chicken and gravy level sprinkle shredded cheese on top and bake at 350 for approximately 20 minutes or until golden and bubbl

In [92]:
with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(corpus_text)

In [94]:
MAX_RECIPES = 1000
MAX_WORDS = 200
SEQ_LEN = 20

# Create input sequences and targets
sequences = []
targets = []

tokenizer = Tokenizer()


for text in recipes[:MAX_RECIPES]:

    # Convert recipe text into word IDs
    words = tokenizer.texts_to_sequences([text])[0]

    # Limit recipe to 200 words
    words = words[:MAX_WORDS]

    # Create 20-word input → next-word target
    for i in range(SEQ_LEN, len(words)):
        sequences.append(words[i-SEQ_LEN:i])
        targets.append(words[i])

# Convert to NumPy arrays
X = np.array(sequences, dtype=np.int32)
y = np.array(targets, dtype=np.int32)

# Check the data
print("Number of samples:", len(X))
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Vocabulary size:", total_words)

AttributeError: 'dict' object has no attribute 'lower'

In [ ]:
y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (157492, 176)
y shape: (157492, 3368)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (125993, 176)
X_test : (31499, 176)
y_train: (125993, 3368)
y_test : (31499, 3368)


In [ ]:
model = Sequential([
    Embedding(
        input_dim=total_words,
        output_dim=128
    ),

    LSTM(150),

    Dropout(0.2),

    Dense(total_words, activation="softmax")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [ ]:
y = to_categorical(y, num_classes=total_words)

MemoryError: Unable to allocate 13.0 TiB for an array with shape (530433056, 3368) and data type float64

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stopping]
)